In [1]:
!pip uninstall -y delta-spark pyspark
!pip install pyspark==4.0.1 delta-spark==4.0.1

Found existing installation: pyspark 4.0.4
Uninstalling pyspark-4.0.4:
  Successfully uninstalled pyspark-4.0.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.2/434.2 MB 1.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.2 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.0.1-py2.py3-none-any.whl size=434813860 sha256=79e82eb1361f2e477b89dbb71d7193cdd4bbc0900d0f43c65accac04f95725ce
  Stored in directory: /root/.cache/pip/wheels/00/e3/92/8594f4cee2c9fd4ad82fe85e4bf2559ab8ea84ef19b1dd3d15
Successfully built pyspark


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("UPI Payment Data Engineering")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

#Step 1: Creating dataframe

#A. Creating UPI Transaction table

In [3]:
data = [

(" txn_2026_100001 ", "upi", "rahul.kumar@okaxis", "9823456712", "1250.50", "SUCCESS", "2026/02/01 09:15:23", "M001", "Amazon", "Delhi"),

("TXN-2026100002", " UPI ", "PRIYA.SHARMA@OKHDFC ", "09876543210", "₹2,499", "success", "01-02-2026 10:22:11", "M002", "Flipkart", " Mumbai "),

("txn2026100003", "upi", "amit.verma@oksbi", "919876543210", "3500", "FAILED", "2026-02-01 11:05:42", "M003", "SWIGGY", "Bangalore"),

("TXN_2026_100004", "UPI", "neha.singh@okicici", "9876543210", "1,250.75", " SUCCESS ", "2026/02/01 12:45:18", "M001", "amazon", "Delhi"),

("txn-2026-100005", "upi", "rohit.mehta@okaxis", "98765-43210", "₹750.00", "failed", "02-02-2026 08:15:09", "M004", "Myntra", "Kolkata"),

(" TXN2026100006", "UPI", "pooja.gupta@okhdfc", "9876543210", "999.99", "PENDING", "2026-02-02 09:31:55", "M005", "BookMyShow", "Chennai"),

("txn_2026_100007", "upi", "vikas.rai@oksbi", "987654321", "125", "SUCCESS", "2026/02/02 14:10:33", "M006", "Swiggy", "Pune"),

("TXN-2026100008", "UPI", "anita.jain@okaxis", "91 9876543210", "5,500", "FAILED", "02-02-2026 16:45:20", "M007", "Flipkart", "Hyderabad"),

("txn2026100009", "UPI", "suresh.patel@okicici", "9876543210", "₹875.25", "success", "2026-02-02 18:20:10", "M008", "Amazon", "Ahmedabad"),

("TXN_2026_100010", "upi", "kiran.das@okhdfc", "9876543210", "1500", "SUCCESS", "2026/02/03 09:05:45", "M009", "Myntra", "Guwahati"),

("txn-2026-100011", "UPI", "meena.roy@oksbi", "9876543210", "2.5K", "SUCCESS", "03-02-2026 11:20:15", "M010", "Swiggy", "Delhi"),

("TXN2026100012", "upi", "arjun.sen@okaxis", "9876543210", "1.2K", "FAILED", "2026-02-03 13:55:42", "M011", "Amazon", "Mumbai"),

("txn_2026_100013", "UPI", "RAVI.KUMAR@OKICICI", "9876543210", "10K", "SUCCESS", "2026/02/04 15:15:30", "M012", "Flipkart", "Bangalore"),

("TXN-2026100014", "upi", "sunita.paul@okhdfc", "9876543210", "999", "PENDING", "04-02-2026 17:40:22", "M013", "Myntra", "Kolkata"),

("txn2026100015", "UPI", "deepak.sharma@oksbi", "9876543210", "₹3,250.75", " SUCCESS", "2026-02-04 10:12:05", "M014", "BookMyShow", "Chennai"),

("TXN_2026_100016", "upi", "riya.agarwal@okaxis", "9876543210", "750.50", "FAILED", "2026/02/04 12:30:44", "M015", "Swiggy", "Pune"),

("txn-2026-100017", "UPI", "manoj.yadav@okicici", "9876543210", "4,500", "SUCCESS", "04-02-2026 14:25:19", "M016", "Amazon", "Hyderabad"),

("TXN2026100018", "upi", "sakshi.mishra@okhdfc", "9876543210", "875", "SUCCESS", "2026-02-05 16:05:37", "M017", "Flipkart", "Ahmedabad"),

("txn_2026_100019", "UPI", "naveen.joshi@oksbi", "9876543210", "1,999", "FAILED", "2026/02/05 18:55:10", "M018", "Myntra", "Guwahati"),

("TXN-2026100020", "upi", "komal.bora@okaxis", "9876543210", "625.50", "SUCCESS", "05-02-2026 20:15:28", "M019", "Swiggy", "Delhi"),

# --- deliberately messy records ---

(" txn_2026_100021 ", "UPI", "rahul.kumar @ okaxis", "98765 43210", "₹1,250.50", "Success", "2026/02/06 09:10:22", "M020", "AMAZON ", " Delhi "),

("TXN-2026100022", "upi", "priya.sharma@okhdfc", "+91-9876543210", "2.75K", "FAILED ", "06-02-2026 10:20:15", "M021", "flipkart", "Mumbai"),

("txn2026100023", "UPI", "amit.verma@oksbi", "98765432100", "500", "COMPLETED", "2026-02-06 11:30:40", "M022", "Swiggy", "Bangalore"),

("TXN_2026_100024", "UPI ", "neha.singh@okicici", "9876543210", "0", "SUCCESS", "2026/02/06 12:15:10", "M023", "Myntra", "Kolkata"),

("txn-2026-100025", "upi", "rohit.mehta@@okaxis", "9876543210", "-750", "FAILED", "2026-02-06 13:45:55", "M024", "Amazon", "Chennai"),

("TXN2026100026", "UPI", "pooja.gupta@okhdfc", "9876543210", "1,250.00", "PEND", "06-02-2026 14:20:33", "M025", "BookMyShow", "Pune"),

("txn_2026_100027", "upi", "vikas.rai@oksbi", "98765-4321", "₹850", "SUCCESS", "2026/02/06 15:10:44", "M026", "Swiggy", "Hyderabad"),

("TXN-2026100028", "upi", "anita.jain@okaxis", None, "900", "FAILED", "2026-02-06 16:30:20", "M027", "Flipkart", "Ahmedabad"),

("txn2026100029", "UPI", None, "9876543210", "1.5K", "SUCCESS", "06-02-2026 17:25:19", "M028", "Amazon", "Guwahati"),

("TXN_2026_100030", None, "suresh.patel@okicici", "9876543210", "2,000", "SUCCESS", "2026/02/07 09:05:15", "M029", "Myntra", "Delhi"),

("txn-2026-100031", "upi", "kiran.das@okhdfc", "9876543210", "₹3.2K", "UNKNOWN", "2026-02-07 10:15:25", "M030", "Swiggy", "Mumbai"),

("TXN2026100032", "UPI", "meena.roy@oksbi", "9876543210", "1,100", "FAILED", "2026-02-07 11:20:30", "M031", "Amazon", "Bangalore"),

("txn_2026_100033", "upi", "arjun.sen@okaxis", "9876543210", "999.99", "SUCCESS", "07/02/2026 12:30:10", "M032", "Flipkart", "Kolkata"),

("TXN-2026100034", "UPI", "sunita.paul@okhdfc", "9876543210", "₹5L", "SUCCESS", "2026/02/07 13:40:20", "M033", "Myntra", "Chennai"),

("txn2026100035", "upi", "deepak.sharma@oksbi", "9876543210", "1.25L", "FAILED", "07-02-2026 14:50:30", "M034", "BookMyShow", "Pune"),

("TXN_2026_100036", "UPI", "riya.agarwal@okaxis", "9876543210", "abc", "SUCCESS", "2026-02-07 15:25:40", "M035", "Swiggy", "Hyderabad"),

("txn-2026-100037", "upi", "manoj.yadav@okicici", "1234567890", "450", "SUCCESS", "2026/02/07 16:35:50", "M999", "Amazon", "Delhi"),

("TXN2026100038", "UPI", "sakshi.mishra@okhdfc", "9876543210", "750", "FAILED", "2026-02-07 17:45:10", "M037", "UnknownStore", "Mumbai"),

("txn_2026_100039", "upi", "naveen.joshi@oksbi", "9876543210", "850", "SUCCESS", "2026/02/07 18:55:20", "M038", "Flipkart", " "),

("TXN-2026100040", "UPI", "komal.bora@okaxis", "9876543210", "1,250.50", "SUCCESS", "2026-02-07 19:15:30", "M039", "Swiggy", "Delhi"),

# Duplicate transaction — deliberately inserted
("txn_2026_100040", "UPI", "komal.bora@okaxis", "9876543210", "1,250.50", "SUCCESS", "2026-02-07 19:15:30", "M039", "Swiggy", "Delhi"),

]

columns = [
    "Transaction_ID",
    "Payment_Method",
    "UPI_ID",
    "Phone",
    "Amount",
    "Transaction_Status",
    "Transaction_Timestamp",
    "Merchant_ID",
    "Merchant_Name",
    "City"
]

upi_transactions_df = spark.createDataFrame(data, columns)

upi_transactions_df.show(50, truncate=False)

+-----------------+--------------+--------------------+--------------+---------+------------------+---------------------+-----------+-------------+---------+
|Transaction_ID   |Payment_Method|UPI_ID              |Phone         |Amount   |Transaction_Status|Transaction_Timestamp|Merchant_ID|Merchant_Name|City     |
+-----------------+--------------+--------------------+--------------+---------+------------------+---------------------+-----------+-------------+---------+
| txn_2026_100001 |upi           |rahul.kumar@okaxis  |9823456712    |1250.50  |SUCCESS           |2026/02/01 09:15:23  |M001       |Amazon       |Delhi    |
|TXN-2026100002   | UPI          |PRIYA.SHARMA@OKHDFC |09876543210   |₹2,499   |success           |01-02-2026 10:22:11  |M002       |Flipkart     | Mumbai  |
|txn2026100003    |upi           |amit.verma@oksbi    |919876543210  |3500     |FAILED            |2026-02-01 11:05:42  |M003       |SWIGGY       |Bangalore|
|TXN_2026_100004  |UPI           |neha.singh@okicici

#B. Merchant_Master table

In [4]:
merchant_master_data = [

    ("M001", "Amazon"),
    ("M002", "Flipkart"),
    ("M003", "Swiggy"),
    ("M004", "Myntra"),
    ("M005", "BookMyShow"),
    ("M006", "Swiggy"),
    ("M007", "Flipkart"),
    ("M008", "Amazon"),
    ("M009", "Myntra"),
    ("M010", "Swiggy"),
    ("M011", "Amazon"),
    ("M012", "Flipkart"),
    ("M013", "Myntra"),
    ("M014", "BookMyShow"),
    ("M015", "Swiggy"),
    ("M016", "Amazon"),
    ("M017", "Flipkart"),
    ("M018", "Myntra"),
    ("M019", "Swiggy"),
    ("M020", "Amazon"),
    ("M021", "Flipkart"),
    ("M022", "Swiggy"),
    ("M023", "Myntra"),
    ("M024", "Amazon"),
    ("M025", "BookMyShow"),
    ("M026", "Swiggy"),
    ("M027", "Flipkart"),
    ("M028", "Amazon"),
    ("M029", "Myntra"),
    ("M030", "Swiggy"),
    ("M031", "Amazon"),
    ("M032", "Flipkart"),
    ("M033", "Myntra"),
    ("M034", "BookMyShow"),
    ("M035", "Swiggy"),
    ("M037", "Flipkart"),
    ("M038", "Flipkart"),
    ("M039", "Swiggy"),

]

merchant_master_df = spark.createDataFrame(merchant_master_data, ["Merchant_ID", "Merchant_Name"])

merchant_master_df.show()

+-----------+-------------+
|Merchant_ID|Merchant_Name|
+-----------+-------------+
|       M001|       Amazon|
|       M002|     Flipkart|
|       M003|       Swiggy|
|       M004|       Myntra|
|       M005|   BookMyShow|
|       M006|       Swiggy|
|       M007|     Flipkart|
|       M008|       Amazon|
|       M009|       Myntra|
|       M010|       Swiggy|
|       M011|       Amazon|
|       M012|     Flipkart|
|       M013|       Myntra|
|       M014|   BookMyShow|
|       M015|       Swiggy|
|       M016|       Amazon|
|       M017|     Flipkart|
|       M018|       Myntra|
|       M019|       Swiggy|
|       M020|       Amazon|
+-----------+-------------+
only showing top 20 rows


#Step 2: Creating Bronze Data

#A. Project path

In [5]:
project_root = "/content/delta/upi_transactions"

bronze_upi_txn_path = f"{project_root}/upi_transactions/bronze"
bronze_merchant_master_path = f"{project_root}/bronze/merchant_master"

#B. Writing into delta

In [6]:
upi_transactions_df.write.format("delta").mode("overwrite").save(bronze_upi_txn_path)
merchant_master_df.write.format("delta").mode("overwrite").save(bronze_merchant_master_path)

#C. Reading back bronze data

In [7]:
bronze_delta_upi_transactions_df = spark.read.format("delta").load(bronze_upi_txn_path)

#Step 4: Silver Layer Transformation

#A. Defining regular expressions

In [8]:
txn_id_format = r"^TXN-20\d{2}-\d{6}$"
upi_id_format = r"^[a-z]+(?:\.[a-z]+)?@[a-z]+"
phone_format = r"^[6-9]\d{9}$"
txn_statuses = r"^(SUCCESS|FAILED|PENDING)$"
merchants_list = [row["Merchant_ID"]
                  for row in merchant_master_df.select("Merchant_ID").collect()]

#B. Cleaning bronze data

In [89]:
silver_upi_transactions_df = (bronze_delta_upi_transactions_df
                              .withColumn("raw_Transaction_ID", sf.col("Transaction_ID"))
                              .withColumn("Transaction_ID",
                                          sf.regexp_replace(
                                              sf.regexp_replace(
                                                  sf.upper(sf.trim(sf.col("Transaction_ID"))),
                                                  r"[-_\s+]", ""), r"^(TXN)(\d{4})(\d{6})$",
                                              r"$1-$2-$3"))
                              .withColumn("valid_Transaction_ID",
                                          sf.coalesce(sf.col("Transaction_ID")
                                          .rlike(txn_id_format), sf.lit(False)))
                              .withColumn("Transaction_ID",
                                          sf.when(sf.col("valid_Transaction_ID"),
                                                  sf.col("Transaction_ID"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Payment_Method", sf.col("Payment_Method"))
                              .withColumn("Payment_Method",
                                          sf.upper(sf.trim(sf.col("Payment_Method"))))
                              .withColumn("valid_Payment_Method",
                                          sf.coalesce(sf.col("Payment_Method")
                                          .isin("UPI"), sf.lit(False)))
                              .withColumn("Payment_Method",
                                          sf.when(sf.col("valid_Payment_Method"),
                                                  sf.col("Payment_Method"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_UPI_ID", sf.col("UPI_ID"))
                              .withColumn("UPI_ID",
                                          sf.regexp_replace(
                                              sf.lower(sf.trim(sf.col("UPI_ID"))),
                                              r"\s+", ""))
                              .withColumn("valid_UPI_ID",
                                          sf.coalesce(sf.col("UPI_ID")
                                          .rlike(upi_id_format), sf.lit(False)))
                              .withColumn("UPI_ID",
                                          sf.when(sf.col("valid_UPI_ID"),
                                                  sf.col("UPI_ID"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Phone", sf.col("Phone"))
                              .withColumn("Phone",
                                          sf.regexp_replace(
                                              sf.regexp_replace(sf.trim("Phone"),
                                                                r"[ -/]", ""),
                                              r"^(?:0091|\+91|91|0)", ""))
                              .withColumn("valid_Phone",
                                          sf.coalesce(sf.col("Phone")
                                          .rlike(phone_format), sf.lit(False)))
                              .withColumn("Phone",
                                          sf.when(sf.col("valid_Phone"),
                                                  sf.col("Phone"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Amount", sf.col("Amount"))
                              .withColumn("Amount",
                                          sf.regexp_replace(
                                              sf.regexp_replace(
                                                  sf.upper(sf.trim("Amount")),
                                                  r"[ /,]", ""),
                                              r"^(?:\$|₹|INR|USD)", ""))
                              .withColumn("Amount",
                                          sf.when(sf.col("Amount").rlike(r"K$"),
                                                  sf.regexp_extract(sf.col("Amount"),
                                                                    r"^(\d+(?:\.\d+)?)K$", 1)
                                                  .try_cast("double")*1000)
                                          .when(sf.col("Amount").rlike(r"(?:LAKH|L)$"),
                                                  sf.regexp_extract(sf.col("Amount"),
                                                                    r"^(\d+(?:\.\d+)?)(?:LAKH|L)$",
                                                                    1).try_cast("double")*100000)
                                          .otherwise(sf.col("Amount").try_cast("double")))
                              .withColumn("valid_Amount",
                                          sf.coalesce(sf.col("Amount") > 0, sf.lit(False)))
                              .withColumn("Amount",
                                          sf.when(sf.col("valid_Amount"),
                                                  sf.col("Amount"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Transaction_Status", sf.col("Transaction_Status"))
                              .withColumn("Transaction_Status",
                                          sf.upper(sf.trim("Transaction_Status")))
                              .withColumn("valid_Transaction_Status",
                                          sf.coalesce(sf.col("Transaction_Status")
                                          .rlike(txn_statuses), sf.lit(False)))
                              .withColumn("Transaction_Status",
                                          sf.when(sf.col("valid_Transaction_Status"),
                                                  sf.col("Transaction_Status"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Transaction_Timestamp",
                                          sf.col("Transaction_Timestamp"))
                              .withColumn("Transaction_Timestamp",
                                          sf.trim("Transaction_Timestamp"))
                              .withColumn("Transaction_Timestamp",
                                          sf.coalesce(
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("yyyy/MM/dd HH:mm:ss")),
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("yyyy-MM-dd HH:mm:ss")),
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("dd-MM-yyyy HH:mm:ss")),
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("MM-dd-yyyy HH:mm:ss")),
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("dd/MM/yyyy HH:mm:ss")),
                                              sf.try_to_timestamp("Transaction_Timestamp",
                                                                  sf.lit("MM/dd/yyyy HH:mm:ss"))))
                              .withColumn("valid_Transaction_Timestamp",
                                          sf.coalesce(sf.col("Transaction_Timestamp").isNotNull(),
                                                      sf.lit(False)))
                              .withColumn("Transaction_Timestamp",
                                          sf.when(sf.col("valid_Transaction_Timestamp"),
                                                  sf.col("Transaction_Timestamp"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Merchant_ID", sf.col("Merchant_ID"))
                              .withColumn("Merchant_ID", sf.upper(sf.trim("Merchant_ID")))
                              .withColumn("valid_Merchant_ID",
                                          sf.coalesce(sf.col("Merchant_ID")
                                          .rlike(r"^M\d{3}$"), sf.lit(False)))
                              .withColumn("Merchant_ID",
                                          sf.when(sf.col("valid_Merchant_ID"),
                                                  sf.col("Merchant_ID"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_Merchant_Name", sf.col("Merchant_Name"))
                              .withColumn("Merchant_Name",
                                          sf.regexp_replace(
                                              sf.initcap(sf.trim("Merchant_Name")), r"\s+", " "))
                              .withColumn("valid_Merchant_Name",
                                          sf.coalesce(sf.col("Merchant_Name")
                                          .rlike(r"^[A-Za-z0-9\s]+$"),
                                                      sf.lit(False)))
                              .withColumn("Merchant_Name",
                                          sf.when(sf.col("valid_Merchant_Name"),
                                                  sf.col("Merchant_Name"))
                                          .otherwise(sf.lit(None)))
                              .withColumn("raw_City", sf.col("City"))
                              .withColumn("City",
                                          sf.regexp_replace(
                                              sf.initcap(sf.trim("City")), r"\s+", " "))
                              .withColumn("valid_City",
                                          sf.coalesce(sf.col("City").rlike(r"^[A-Za-z0-9\s]+$"),
                                                      sf.lit(False)))
                              .withColumn("City",
                                          sf.when(sf.col("valid_City"), sf.col("City"))
                                          .otherwise(sf.lit(None)))
                              )

#D. Creating flags for valid and invalid records

In [90]:
silver_upi_transactions_df = (silver_upi_transactions_df
                              .withColumn("Validation_Failed",
                                          ~(sf.col("valid_Transaction_ID") &
                                            sf.col("valid_Payment_Method") &
                                            sf.col("valid_UPI_ID") &
                                            sf.col("valid_Phone") &
                                            sf.col("valid_Amount") &
                                            sf.col("valid_Transaction_Status") &
                                            sf.col("valid_Transaction_Timestamp") &
                                            sf.col("valid_Merchant_ID") &
                                            sf.col("valid_Merchant_Name") &
                                            sf.col("valid_City")))
                              )

In [91]:
(silver_upi_transactions_df
 .select("valid_Transaction_ID", "valid_Payment_Method", "valid_UPI_ID",
         "valid_Phone", "valid_Amount", "valid_Transaction_Status",
         "valid_Transaction_Timestamp", "valid_Merchant_ID",
         "valid_Merchant_Name", "valid_City", "Validation_Failed")
 .show(60)
 )

+--------------------+--------------------+------------+-----------+------------+------------------------+---------------------------+-----------------+-------------------+----------+-----------------+
|valid_Transaction_ID|valid_Payment_Method|valid_UPI_ID|valid_Phone|valid_Amount|valid_Transaction_Status|valid_Transaction_Timestamp|valid_Merchant_ID|valid_Merchant_Name|valid_City|Validation_Failed|
+--------------------+--------------------+------------+-----------+------------+------------------------+---------------------------+-----------------+-------------------+----------+-----------------+
|                true|                true|        true|       true|        true|                    true|                       true|             true|               true|      true|            false|
|                true|                true|        true|       true|        true|                    true|                       true|             true|               true|      true|         

#Step 5: Quarantine transaction table

#A. Quarantine Transaction_ID

In [92]:
quarantined_transaction_id_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Transaction_ID"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Transaction_ID").alias("Column_Name"),
                                      sf.col("raw_Transaction_ID").alias("Invalid_Value"),
                                      sf.lit("DQ001").alias("Rule_ID"),
                                      sf.lit("Invalid Transaction_ID/format")
                                      .alias("Failure_Reason"))
                              )

In [93]:
quarantined_transaction_id_df.show()

+--------------+-----------+-------------+-------+--------------+
|Transaction_ID|Column_Name|Invalid_Value|Rule_ID|Failure_Reason|
+--------------+-----------+-------------+-------+--------------+
+--------------+-----------+-------------+-------+--------------+



#B. Quarantine Payment_Mwthod

In [94]:
quarantined_payment_method_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Payment_Method"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Payment_Method").alias("Column_Name"),
                                      sf.col("raw_Payment_Method").alias("Invalid_Value"),
                                      sf.lit("DQ002").alias("Rule_ID"),
                                      sf.lit("Invalid Payment_Method/format")
                                      .alias("Failure_Reason"))
                              )

In [95]:
quarantined_payment_method_df.show()

+---------------+--------------+-------------+-------+--------------------+
| Transaction_ID|   Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+--------------+-------------+-------+--------------------+
|TXN_2026_100030|Payment_Method|         NULL|  DQ002|Invalid Payment_M...|
+---------------+--------------+-------------+-------+--------------------+



#C. Quarantine UPI_ID

In [96]:
quarantined_upi_id_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_UPI_ID"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("UPI_ID").alias("Column_Name"),
                                      sf.col("raw_UPI_ID").alias("Invalid_Value"),
                                      sf.lit("DQ003").alias("Rule_ID"),
                                      sf.lit("Invalid UPI_ID/format")
                                      .alias("Failure_Reason"))
                              )

In [97]:
quarantined_upi_id_df.show()

+---------------+-----------+-------------------+-------+--------------------+
| Transaction_ID|Column_Name|      Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+-----------+-------------------+-------+--------------------+
|txn-2026-100025|     UPI_ID|rohit.mehta@@okaxis|  DQ003|Invalid UPI_ID/fo...|
|  txn2026100029|     UPI_ID|               NULL|  DQ003|Invalid UPI_ID/fo...|
+---------------+-----------+-------------------+-------+--------------------+



#D. Quarantine Phone

In [98]:
quarantined_phone_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Phone"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Phone").alias("Column_Name"),
                                      sf.col("raw_Phone").alias("Invalid_Value"),
                                      sf.lit("DQ004").alias("Rule_ID"),
                                      sf.lit("Invalid Phone/format")
                                      .alias("Failure_Reason"))
                              )

In [99]:
quarantined_phone_df.show()

+---------------+-----------+-------------+-------+--------------------+
| Transaction_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+-----------+-------------+-------+--------------------+
|  txn2026100023|      Phone|  98765432100|  DQ004|Invalid Phone/format|
|txn_2026_100027|      Phone|   98765-4321|  DQ004|Invalid Phone/format|
| TXN-2026100028|      Phone|         NULL|  DQ004|Invalid Phone/format|
|txn-2026-100037|      Phone|   1234567890|  DQ004|Invalid Phone/format|
|txn_2026_100007|      Phone|    987654321|  DQ004|Invalid Phone/format|
+---------------+-----------+-------------+-------+--------------------+



#E. Quarantine Amount

In [100]:
quarantined_amount_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Amount"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Amount").alias("Column_Name"),
                                      sf.col("raw_Amount").alias("Invalid_Value"),
                                      sf.lit("DQ005").alias("Rule_ID"),
                                      sf.lit("Invalid Amount/format")
                                      .alias("Failure_Reason"))
                              )

In [101]:
quarantined_amount_df.show()

+---------------+-----------+-------------+-------+--------------------+
| Transaction_ID|Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+-----------+-------------+-------+--------------------+
|TXN_2026_100024|     Amount|            0|  DQ005|Invalid Amount/fo...|
|txn-2026-100025|     Amount|         -750|  DQ005|Invalid Amount/fo...|
|TXN_2026_100036|     Amount|          abc|  DQ005|Invalid Amount/fo...|
+---------------+-----------+-------------+-------+--------------------+



#F. Quarantine Transaction_Status

In [102]:
quarantined_transaction_status_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Transaction_Status"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Transaction_Status").alias("Column_Name"),
                                      sf.col("raw_Transaction_Status").alias("Invalid_Value"),
                                      sf.lit("DQ006").alias("Rule_ID"),
                                      sf.lit("Invalid Transaction_Status/format")
                                      .alias("Failure_Reason"))
                              )

In [103]:
quarantined_transaction_status_df.show()

+---------------+------------------+-------------+-------+--------------------+
| Transaction_ID|       Column_Name|Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+------------------+-------------+-------+--------------------+
|  txn2026100023|Transaction_Status|    COMPLETED|  DQ006|Invalid Transacti...|
|  TXN2026100026|Transaction_Status|         PEND|  DQ006|Invalid Transacti...|
|txn-2026-100031|Transaction_Status|      UNKNOWN|  DQ006|Invalid Transacti...|
+---------------+------------------+-------------+-------+--------------------+



#G. Quarantine Merchant_ID

In [104]:
quarantined_merchant_id_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Merchant_ID"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Merchant_ID").alias("Column_Name"),
                                      sf.col("raw_Merchant_ID").alias("Invalid_Value"),
                                      sf.lit("DQ007 and DQ010").alias("Rule_ID"),
                                      sf.lit("Invalid Merchant_ID/Format")
                                      .alias("Failure_Reason"))
                              )

In [105]:
quarantined_merchant_id_df.show()

+--------------+-----------+-------------+-------+--------------+
|Transaction_ID|Column_Name|Invalid_Value|Rule_ID|Failure_Reason|
+--------------+-----------+-------------+-------+--------------+
+--------------+-----------+-------------+-------+--------------+



#H. Quarantine Merchant_Name

In [106]:
quarantined_merchant_name_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_Merchant_Name"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("Merchant_Name").alias("Column_Name"),
                                      sf.col("raw_Merchant_Name").alias("Invalid_Value"),
                                      sf.lit("DQ008").alias("Rule_ID"),
                                      sf.lit("Invalid Merchant_Name/format")
                                      .alias("Failure_Reason"))
                              )

In [107]:
quarantined_merchant_name_df.show()

+--------------+-----------+-------------+-------+--------------+
|Transaction_ID|Column_Name|Invalid_Value|Rule_ID|Failure_Reason|
+--------------+-----------+-------------+-------+--------------+
+--------------+-----------+-------------+-------+--------------+



#I. Quarantine City

In [108]:
quarantined_city_df = (silver_upi_transactions_df
                              .filter(~sf.col("valid_City"))
                              .select(sf.col("raw_Transaction_ID").alias("Transaction_ID"),
                                      sf.lit("City").alias("Column_Name"),
                                      sf.col("raw_City").alias("Invalid_Value"),
                                      sf.lit("DQ009").alias("Rule_ID"),
                                      sf.lit("Invalid City/format")
                                      .alias("Failure_Reason"))
                              )

In [109]:
quarantined_city_df.show()

+---------------+-----------+-------------+-------+-------------------+
| Transaction_ID|Column_Name|Invalid_Value|Rule_ID|     Failure_Reason|
+---------------+-----------+-------------+-------+-------------------+
|txn_2026_100039|       City|             |  DQ009|Invalid City/format|
+---------------+-----------+-------------+-------+-------------------+



#J. Quarantine duplicate transactions

In [112]:
silver_upi_transactions_df = (silver_upi_transactions_df
                              .withColumn("Row_Number",
                                          sf.row_number().over(Window
                                                               .partitionBy("Transaction_ID")
                                                               .orderBy("Transaction_ID")))
                              )

In [113]:
quarantined_duplicate_transaction_df = (silver_upi_transactions_df
                                   .filter("Row_Number > 1")
                                   .select("Transaction_ID",
                                           sf.lit("Transaction_ID").alias("Column_Name"),
                                           sf.col("Transaction_ID").alias("Invalid_Value"),
                                           sf.lit("DQ009").alias("Rule_ID"),
                                           sf.lit("Duplicate Transaction_ID")
                                           .alias("Failure_Reason"))
                                   )

In [114]:
quarantined_duplicate_transaction_df.show()

+---------------+--------------+---------------+-------+--------------------+
| Transaction_ID|   Column_Name|  Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+--------------+---------------+-------+--------------------+
|TXN-2026-100040|Transaction_ID|TXN-2026-100040|  DQ009|Duplicate Transac...|
+---------------+--------------+---------------+-------+--------------------+



#K. Quarantine orphan merchants

In [123]:
silver_upi_transactions_df = (silver_upi_transactions_df
                                   .join(merchant_master_df
                                         .select("Merchant_ID", sf.col("Merchant_Name")
                                         .alias("Merchant_Name_Master")),
                                         on="Merchant_ID",
                                         how="left")
                                   )

In [130]:
silver_upi_transactions_df = (silver_upi_transactions_df
                              .withColumn("is_Orphan",
                                          sf.col("Merchant_Name_Master")
                                          .isNull()))

In [131]:
quarantined_orphan_merchants_df = (silver_upi_transactions_df
                                   .filter(sf.col("is_Orphan"))
                                   .select("Transaction_ID",
                                           sf.lit("Merchant_ID").alias("Column_Name"),
                                           sf.col("Merchant_ID").alias("Invalid_Value"),
                                           sf.lit("DQ010").alias("Rule_ID"),
                                           sf.lit("Orphan Merchant_ID")
                                           .alias("Failure_Reason"))
                                   )

In [132]:
quarantined_orphan_merchants_df.show()

+---------------+-----------+-------------+-------+------------------+
| Transaction_ID|Column_Name|Invalid_Value|Rule_ID|    Failure_Reason|
+---------------+-----------+-------------+-------+------------------+
|TXN-2026-100037|Merchant_ID|         M999|  DQ010|Orphan Merchant_ID|
+---------------+-----------+-------------+-------+------------------+



#L. Combined quarantined table

In [133]:
quarantined_upi_transactions_df = (quarantined_transaction_id_df
                                   .unionByName(quarantined_payment_method_df)
                                   .unionByName(quarantined_upi_id_df)
                                   .unionByName(quarantined_phone_df)
                                   .unionByName(quarantined_amount_df)
                                   .unionByName(quarantined_transaction_status_df)
                                   .unionByName(quarantined_merchant_id_df)
                                   .unionByName(quarantined_merchant_name_df)
                                   .unionByName(quarantined_city_df)
                                   .unionByName(quarantined_duplicate_transaction_df)
                                   .unionByName(quarantined_orphan_merchants_df)
                                   )

In [134]:
quarantined_upi_transactions_df.show()

+---------------+------------------+-------------------+-------+--------------------+
| Transaction_ID|       Column_Name|      Invalid_Value|Rule_ID|      Failure_Reason|
+---------------+------------------+-------------------+-------+--------------------+
|TXN_2026_100030|    Payment_Method|               NULL|  DQ002|Invalid Payment_M...|
|txn-2026-100025|            UPI_ID|rohit.mehta@@okaxis|  DQ003|Invalid UPI_ID/fo...|
|  txn2026100029|            UPI_ID|               NULL|  DQ003|Invalid UPI_ID/fo...|
|  txn2026100023|             Phone|        98765432100|  DQ004|Invalid Phone/format|
|txn_2026_100027|             Phone|         98765-4321|  DQ004|Invalid Phone/format|
| TXN-2026100028|             Phone|               NULL|  DQ004|Invalid Phone/format|
|txn-2026-100037|             Phone|         1234567890|  DQ004|Invalid Phone/format|
|txn_2026_100007|             Phone|          987654321|  DQ004|Invalid Phone/format|
|TXN_2026_100024|            Amount|                  

#Step 5: Finalizing cleaned silver table

In [136]:
final_silver_upi_transaction_df = (silver_upi_transactions_df
                                         .filter(~sf.col("Validation_Failed") &
                                                 (sf.col("Row_Number") == 1) &
                                                 ~sf.col("is_Orphan"))
                                         .select("Transaction_ID", "Payment_Method",
                                                 "UPI_ID", "Phone", "Amount",
                                                 "Transaction_Status", "Transaction_Timestamp",
                                                 "Merchant_ID", "Merchant_Name", "City")
                                         )

In [138]:
final_silver_upi_transaction_df.show(60)

+---------------+--------------+--------------------+----------+--------+------------------+---------------------+-----------+-------------+---------+
| Transaction_ID|Payment_Method|              UPI_ID|     Phone|  Amount|Transaction_Status|Transaction_Timestamp|Merchant_ID|Merchant_Name|     City|
+---------------+--------------+--------------------+----------+--------+------------------+---------------------+-----------+-------------+---------+
|TXN-2026-100004|           UPI|  neha.singh@okicici|9876543210| 1250.75|           SUCCESS|  2026-02-01 12:45:18|       M001|       Amazon|    Delhi|
|TXN-2026-100001|           UPI|  rahul.kumar@okaxis|9823456712|  1250.5|           SUCCESS|  2026-02-01 09:15:23|       M001|       Amazon|    Delhi|
|TXN-2026-100002|           UPI| priya.sharma@okhdfc|9876543210|  2499.0|           SUCCESS|  2026-02-01 10:22:11|       M002|     Flipkart|   Mumbai|
|TXN-2026-100003|           UPI|    amit.verma@oksbi|9876543210|  3500.0|            FAILED|  

#Step 6: Data reconcillation

In [141]:
silver_upi_transactions_df.count()

41

In [142]:
final_silver_upi_transaction_df.count()

27

In [143]:
quarantined_upi_transactions_df.count()

17

In [140]:
silver_upi_transactions_df.printSchema()
final_silver_upi_transaction_df.printSchema()
quarantined_upi_transactions_df.printSchema()

root
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- UPI_ID: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Status: string (nullable = true)
 |-- Transaction_Timestamp: timestamp (nullable = true)
 |-- Merchant_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- raw_Transaction_ID: string (nullable = true)
 |-- valid_Transaction_ID: boolean (nullable = false)
 |-- raw_Payment_Method: string (nullable = true)
 |-- valid_Payment_Method: boolean (nullable = false)
 |-- raw_UPI_ID: string (nullable = true)
 |-- valid_UPI_ID: boolean (nullable = false)
 |-- raw_Phone: string (nullable = true)
 |-- valid_Phone: boolean (nullable = false)
 |-- raw_Amount: string (nullable = true)
 |-- valid_Amount: boolean (nullable = false)
 |-- raw_Transaction_Status: string (nullable = true)
 |-- valid_Transaction_Status: boole

In [144]:
final_silver_upi_transaction_df.show(30)

+---------------+--------------+--------------------+----------+--------+------------------+---------------------+-----------+-------------+---------+
| Transaction_ID|Payment_Method|              UPI_ID|     Phone|  Amount|Transaction_Status|Transaction_Timestamp|Merchant_ID|Merchant_Name|     City|
+---------------+--------------+--------------------+----------+--------+------------------+---------------------+-----------+-------------+---------+
|TXN-2026-100004|           UPI|  neha.singh@okicici|9876543210| 1250.75|           SUCCESS|  2026-02-01 12:45:18|       M001|       Amazon|    Delhi|
|TXN-2026-100001|           UPI|  rahul.kumar@okaxis|9823456712|  1250.5|           SUCCESS|  2026-02-01 09:15:23|       M001|       Amazon|    Delhi|
|TXN-2026-100002|           UPI| priya.sharma@okhdfc|9876543210|  2499.0|           SUCCESS|  2026-02-01 10:22:11|       M002|     Flipkart|   Mumbai|
|TXN-2026-100003|           UPI|    amit.verma@oksbi|9876543210|  3500.0|            FAILED|  

#Step 7: Creating Silver delta

In [162]:
print(project_root)

/content/delta/upi_transactions


#A. Creating silver path

In [163]:
silver_delta_cleaned_path = f"{project_root}/silver/cleaned"
silver_delta_final_path = f"{project_root}/silver/final"
silver_delta_quarantined_path = f"{project_root}/silver/quarantined"

#B. Storing silver data into delta

In [164]:
silver_upi_transactions_df.write.format("delta").mode("overwrite").save(silver_delta_cleaned_path)
final_silver_upi_transaction_df.write.format("delta").mode("overwrite").save(silver_delta_final_path)
quarantined_upi_transactions_df.write.format("delta").mode("overwrite").save(silver_delta_quarantined_path)

#C. Reading silver delta data back

In [165]:
final_delta_upi_transactions_df = (spark.read
                                     .format("delta")
                                     .load(silver_delta_final_path)
                                     )

In [166]:
final_delta_upi_transactions_df.show()

+---------------+--------------+--------------------+----------+-------+------------------+---------------------+-----------+-------------+---------+
| Transaction_ID|Payment_Method|              UPI_ID|     Phone| Amount|Transaction_Status|Transaction_Timestamp|Merchant_ID|Merchant_Name|     City|
+---------------+--------------+--------------------+----------+-------+------------------+---------------------+-----------+-------------+---------+
|TXN-2026-100004|           UPI|  neha.singh@okicici|9876543210|1250.75|           SUCCESS|  2026-02-01 12:45:18|       M001|       Amazon|    Delhi|
|TXN-2026-100001|           UPI|  rahul.kumar@okaxis|9823456712| 1250.5|           SUCCESS|  2026-02-01 09:15:23|       M001|       Amazon|    Delhi|
|TXN-2026-100002|           UPI| priya.sharma@okhdfc|9876543210| 2499.0|           SUCCESS|  2026-02-01 10:22:11|       M002|     Flipkart|   Mumbai|
|TXN-2026-100003|           UPI|    amit.verma@oksbi|9876543210| 3500.0|            FAILED|  2026-02

#Step 8: Gold Layer Transformation

#A. Gold transaction-level table

In [167]:
gold_upi_transactions_df = (final_delta_upi_transactions_df
                            .withColumn("Transaction_Date",
                                        sf.to_date("Transaction_Timestamp"))
                            .withColumn("Transaction_Year",
                                        sf.year("Transaction_Timestamp"))
                            .withColumn("Transaction_Month",
                                        sf.month("Transaction_Timestamp"))
                            .withColumn("Transaction_Day",
                                        sf.dayofmonth("Transaction_Timestamp"))
                            .select("Transaction_ID", "Transaction_Date",
                                    "Transaction_Timestamp", "Payment_Method",
                                    "UPI_ID", "Phone", "Amount",
                                    "Transaction_Status", "Merchant_ID",
                                    "Merchant_Name", "City", "Transaction_Year",
                                    "Transaction_Month", "Transaction_Day")
                            )

In [168]:
gold_upi_transactions_df.printSchema()

gold_upi_transactions_df.show(30, False)

root
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: date (nullable = true)
 |-- Transaction_Timestamp: timestamp (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- UPI_ID: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- Transaction_Status: string (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Merchant_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Transaction_Year: integer (nullable = true)
 |-- Transaction_Month: integer (nullable = true)
 |-- Transaction_Day: integer (nullable = true)

+---------------+----------------+---------------------+--------------+--------------------+----------+--------+------------------+-----------+-------------+---------+----------------+-----------------+---------------+
|Transaction_ID |Transaction_Date|Transaction_Timestamp|Payment_Method|UPI_ID              |Phone     |Amount  |Transaction_Status|Merchant_ID|Merc

#B. Merchant Performance Gold

In [171]:
gold_merchant_summary_df = (gold_upi_transactions_df
                            .groupBy("Merchant_ID", "Merchant_Name", "City")
                            .agg(
                                sf.count("Transaction_ID").alias("Total_Transactions"),
                                sf.sum(
                                    sf.when(sf.col("Transaction_Status") == "SUCCESS", 1)
                                    .otherwise(0)).alias("Successful_Transactions"),
                                sf.sum(
                                    sf.when(sf.col("Transaction_Status") == "FAILED", 1)
                                    .otherwise(0)).alias("Failed_Transactions"),
                                sf.sum(
                                    sf.when(sf.col("Transaction_Status") == "PENDING", 1)
                                    .otherwise(0)).alias("Pending_Transactions"),
                                sf.sum("Amount").alias("Total_Transaction_Amount"),
                                sf.sum(
                                    sf.when(sf.col("Transaction_Status") == "SUCCESS",
                                            sf.col("Amount"))
                                    .otherwise(0)).alias("Successful_Transaction_Amount"),
                                sf.avg("Amount").alias("Average_Transaction_Amount"))
                            )

In [173]:
gold_merchant_summary_df.show(50)

+-----------+-------------+---------+------------------+-----------------------+-------------------+--------------------+------------------------+-----------------------------+--------------------------+
|Merchant_ID|Merchant_Name|     City|Total_Transactions|Successful_Transactions|Failed_Transactions|Pending_Transactions|Total_Transaction_Amount|Successful_Transaction_Amount|Average_Transaction_Amount|
+-----------+-------------+---------+------------------+-----------------------+-------------------+--------------------+------------------------+-----------------------------+--------------------------+
|       M013|       Myntra|  Kolkata|                 1|                      0|                  0|                   1|                   999.0|                          0.0|                     999.0|
|       M002|     Flipkart|   Mumbai|                 1|                      1|                  0|                   0|                  2499.0|                       2499.0|        

#C. Daily Payment Summary

In [156]:
gold_daily_payment_summary_df = (gold_upi_transactions_df
                                 .groupBy("Transaction_Date")
                                 .agg(sf.count("Transaction_ID")
                                 .alias("Total_Transactions"),
                                      sf.sum(
                                          sf.when(
                                              sf.col("Transaction_Status") == "SUCCESS", 1)
                                          .otherwise(0)).alias("Successful_Transactions"),
                                      sf.sum(
                                          sf.when(
                                              sf.col("Transaction_Status") == "FAILED", 1)
                                          .otherwise(0)).alias("Failed_Transactions"),
                                      sf.sum(
                                          sf.when(
                                              sf.col("Transaction_Status") == "PENDING", 1)
                                          .otherwise(0)).alias("Pending_Transactions"),
                                      sf.sum("Amount").alias("Total_Amount"),
                                      sf.sum(
                                          sf.when(
                                              sf.col("Transaction_Status") == "SUCCESS",
                                              sf.col("Amount"))
                                          .otherwise(0)).alias("Successful_Amount"),
                                      sf.avg("Amount").alias("Average_Transaction_Amount"))
                                 .orderBy("Transaction_Date")
                                 )

In [157]:
gold_daily_payment_summary_df.printSchema()
gold_daily_payment_summary_df.show(50)

root
 |-- Transaction_Date: date (nullable = true)
 |-- Total_Transactions: long (nullable = false)
 |-- Successful_Transactions: long (nullable = true)
 |-- Failed_Transactions: long (nullable = true)
 |-- Pending_Transactions: long (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Successful_Amount: double (nullable = true)
 |-- Average_Transaction_Amount: double (nullable = true)

+----------------+------------------+-----------------------+-------------------+--------------------+------------+-----------------+--------------------------+
|Transaction_Date|Total_Transactions|Successful_Transactions|Failed_Transactions|Pending_Transactions|Total_Amount|Successful_Amount|Average_Transaction_Amount|
+----------------+------------------+-----------------------+-------------------+--------------------+------------+-----------------+--------------------------+
|      2026-02-01|                 4|                      3|                  1|                   0|     8500.2

#D. Add success rate

In [158]:
gold_merchant_summary_df = (gold_merchant_summary_df
                            .withColumn("Success_Rate",
                                        sf.round(
                                            sf.col("Successful_Transactions")
                                            / sf.col("Total_Transactions") * 100, 2))
                            )

In [161]:
gold_merchant_summary_df.show(50)

+-----------+-------------+---------+------------------+-----------------------+-------------------+--------------------+------------------------+-----------------------------+--------------------------+------------+
|Merchant_ID|Merchant_Name|     City|Total_Transactions|Successful_Transactions|Failed_Transactions|Pending_Transactions|Total_Transaction_Amount|Successful_Transaction_Amount|Average_Transaction_Amount|Success_Rate|
+-----------+-------------+---------+------------------+-----------------------+-------------------+--------------------+------------------------+-----------------------------+--------------------------+------------+
|       M013|       Myntra|  Kolkata|                 1|                      0|                  0|                   1|                   999.0|                          0.0|                     999.0|         0.0|
|       M002|     Flipkart|   Mumbai|                 1|                      1|                  0|                   0|           

#Step 9: Writing gold data into delta

#A. Gold paths

In [175]:
gold_delta_transactions_path = f"{project_root}/gold/transactions"
gold_delta_merchant_summary_path = f"{project_root}/gold/merchant_summary"
gold_delta_daily_payment_summary_path = f"{project_root}/gold/daily_payment_summary"

#B. Writing into delta

In [176]:
gold_upi_transactions_df.write.format("delta").mode("overwrite").save(gold_delta_transactions_path)
gold_merchant_summary_df.write.format("delta").mode("overwrite").save(gold_delta_merchant_summary_path)
gold_daily_payment_summary_df.write.format("delta").mode("overwrite").save(gold_delta_daily_payment_summary_path)